In [1]:
using HDF5
using Unitful
import PhysicalConstants.CODATA2018: m_n, e, ħ
using FileIO
using GeometryBasics
using LinearAlgebra
using BenchmarkTools
using MeshIO
using SparseArrays
using StaticArrays
using LoopVectorization

In [2]:
# Extracting the contents of the .nxspe file.

en_i, azi, pol, data, Δen = h5open("test_nxspe_data/LET104215_3.7meV_1to1.nxspe", "r") do f
    # Initial energy in meV.
    en_i = read(f["ws_out/NXSPE_info/fixed_energy"])[1]
    # Azimuthal angles in degrees.
    azi = read(f["ws_out/data/azimuthal"])
    # Polar angles in degrees.
    pol = read(f["ws_out/data/polar"])
    # Measured signal for each energy bin and each detector.
    data = read(f["ws_out/data/data"])
    # Neutron energy changes, in meV.
    Δen = read(f["ws_out/data/energy"])
    return en_i, azi, pol, data, Δen
end

(3.7, [-137.16071701049805, -137.39026260375977, -137.62151336669922, -137.8544692993164, -138.08910751342773, -138.32544326782227, -138.56351470947266, -138.80327224731445, -139.0447883605957, -139.2881088256836  …  41.245439529418945, 41.48468208312988, 41.7221097946167, 41.9577579498291, 42.19169521331787, 42.42388725280762, 42.65436553955078, 42.883137702941895, 43.110212326049805, 43.33559226989746], [48.28250598907471, 48.177175521850586, 48.07186985015869, 47.9666051864624, 47.8613977432251, 47.75627422332764, 47.651217460632324, 47.546268463134766, 47.44140434265137, 47.33662223815918  …  131.6914520263672, 131.5868377685547, 131.4821891784668, 131.37750625610352, 131.27277374267578, 131.16801834106445, 131.06324005126953, 130.95844650268555, 130.8536720275879, 130.74888229370117], [NaN NaN … NaN NaN; NaN NaN … NaN NaN; … ; NaN NaN … NaN NaN; NaN NaN … NaN NaN], [-2.9600000000000004, -2.9415000000000004, -2.9230000000000005, -2.9045000000000005, -2.8860000000000006, -2.86750000

In [ ]:
# Defining a function to calculate the magnitude of the wavevector, in Angstrom^-1, of the neutron from its energy.

"""
Calculates the magnitude of the wavevector, in Angstrom^-1, from an energy, in meV.

Parameters
----------
en (float): Energy, in meV.

Returns
-------
mag_k (float): Magnitude of wavevector, in Angstrom^-1.
"""
function magk_calc(en :: Float64)
    mag_k = sqrt(2 * m_n * en * e * (1e-3)) / (ħ * (1e10))
    return ustrip(mag_k)
end

@benchmark magk_calc(en_i)

BenchmarkTools.Trial: 10000 samples with 10 evaluations per sample.
 Range (min … max):  1.230 μs … 60.700 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     1.560 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   1.865 μs ±  1.123 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▆██▇▄▃▄▃▃▃▆▅▅▄▅▅▄▄▃▃▂▂▁        ▂           ▁     ▁         ▂
  ███████████████████████▇██▇▇▆▇███▇▇▇▆▇██▇▇████▇▇██▇█▇▅▆▆▅▅ █
  1.23 μs      Histogram: log(frequency) by time     4.66 μs <

 Memory estimate: 840 bytes, allocs estimate: 20.

In [4]:
# Calculating ki, n_bins, n_detectors from the extracted content of the .nxspe file.


# Finding the initial wavevector, in Angstrom^-1, from the initial energy.
ki = zeros(3)
ki[1] = magk_calc(en_i)
# Converting ki to a static array.
ki = SVector{3}(ki)

# Extracting the number of energy bins and number of detectors.
const n_bins = length(Δen) - 1
const n_detectors = length(azi)

98304

In [6]:
# Defining a function to calculate the final energy of neutrons from each bin, based on Ei and the energy change.

"""
Calculates the final neutron energy, in meV, for each energy bin.

Parameters
----------
en_i (float): Pre-scattering neutron energy, in meV.
Δen ((n_bins + 1)-vector with float elements): Neutron energy changes, in meV.

Returns
-------
ef_bins (n_bins-vector with float elements): Post-scattering neuton energy for each bin, in meV.
"""
function ef_calc(en_i :: Float64, Δen :: AbstractVector{Float64})
    ef_bins = zeros(n_bins)
    for i in 1:n_bins
        # Finding the bin centres by averaging the energies on each end of the bin.
        # Determining the final neutron energy, Ef = Ei - Δen based on which energy bin we are considering.
        ef_bins[i] = en_i - ((Δen[i] + Δen[i+1]) / 2)
    end
    # Returning the final energies in a static array.
    return SVector{n_bins}(ef_bins)
end

@benchmark ef_calc(en_i, Δen)

BenchmarkTools.Trial: 10000 samples with 127 evaluations per sample.
 Range (min … max):  581.102 ns … 582.057 μs  ┊ GC (min … max):  0.00% … 99.60%
 Time  (median):       1.083 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):     2.718 μs ±   9.361 μs  ┊ GC (mean ± σ):  26.84% ± 13.09%

  █▂                                                             
  ██▃▂▂▃▄▃▄▄▃▃▂▂▂▂▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▂▂▁▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂ ▂
  581 ns           Histogram: frequency by time           26 μs <

 Memory estimate: 5.12 KiB, allocs estimate: 3.

In [7]:
# Calculating the final neutron energy, in meV, for each energy bin.

ef_bins = ef_calc(en_i, Δen)

320-element SVector{320, Float64} with indices SOneTo(320):
 6.65075
 6.632250000000001
 6.6137500000000005
 6.595250000000001
 6.5767500000000005
 6.558250000000001
 6.539750000000001
 6.521250000000001
 6.502750000000001
 6.484250000000001
 ⋮
 0.8972500000000094
 0.8787500000000099
 0.8602500000000095
 0.84175000000001
 0.8232500000000096
 0.8047500000000101
 0.7862500000000097
 0.7677500000000101
 0.7492500000000049

In [8]:
# Defining the function that calculates the final neutron wavevector from the detector angles and final neutron energies.

"""
Calculates the components of the post-scattering neutron wavevector, in Angstrom^-1, for each detector and for each energy bin.

Parameters
----------
ef_bins (n_bins-vector with float elements): Post-scattering neutron energy of each bin, in meV.
pol (n_detectors-vector with float elements): Polar angles of each detector, in degrees.
azi (n_detectors-vector with float elements): Azimuthal angles of each detector, in degrees.

Returns
-------
kx (n_bins x n_detectors matrix of floats): Post-scattering neutron wavevector component in x direction, in Angstrom^-1.
ky (n_bins x n_detectors matrix of floats): Post-scattering neutron wavevector component in y direction, in Angstrom^-1.
kz (n_bins x n_detectors matrix of floats): Post-scattering neutron wavevector component in z direction, in Angstrom^-1.
"""
function kf_calc(ef_bins :: SVector{n_bins, Float64}, pol :: AbstractVector{Float64}, azi :: AbstractVector{Float64})
    mag_kf = magk_calc.(ef_bins)
    # Reshaping the arrays to allow for broadcasting.
    pol_col = reshape(pol, :, 1)
    azi_col = reshape(azi, :, 1)
    mag_kf_row = reshape(mag_kf, 1, :)
    # Determing the components of the final wavevector using broadcasting.
    kx = mag_kf_row .* (sin.(deg2rad.(pol_col)) .* cos.(deg2rad.(azi_col)))
    ky = mag_kf_row .* (sin.(deg2rad.(pol_col)) .* sin.(deg2rad.(azi_col)))
    kz = mag_kf_row .* cos.(deg2rad.(pol_col))
    # Reshaping these final wavevector grids to align with the grid of data.
    kx = transpose(kx)
    ky = transpose(ky)
    kz = transpose(kz)
    return kx, ky, kz
end

@benchmark kf_calc(ef_bins, pol, azi)

BenchmarkTools.Trial: 3 samples with 1 evaluation per sample.
 Range (min … max):  1.592 s …    2.417 s  ┊ GC (min … max): 0.11% … 0.07%
 Time  (median):     1.873 s               ┊ GC (median):    0.10%
 Time  (mean ± σ):   1.960 s ± 419.370 ms  ┊ GC (mean ± σ):  4.92% ± 8.76%

  █                  █                                     █  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  1.59 s         Histogram: frequency by time         2.42 s <

 Memory estimate: 720.26 MiB, allocs estimate: 6412.

In [9]:
# Calculating the final wavevector, in Angstrom^-1, for each energy bin and each detector.

kx, ky, kz = kf_calc(ef_bins, pol, azi)

([-0.9805723528957592 -0.9825925959589944 … 0.9892722789861215 0.9871790916414052; -0.9792076029610514 -0.9812250342719662 … 0.9878954205889136 0.9858051465195753; … ; -0.33316106768660314 -0.33384746918869146 … 0.3361169705901472 0.33540578540470495; -0.3291226024679551 -0.3298006836443479 … 0.3320426749212351 0.3313401104808738], [-0.9092695375514454 -0.9038484510341793 … 0.9260754147817433 0.9314290044594985; -0.9080040260995464 -0.9025904845915279 … 0.9247865130927191 0.9301326517025937; … ; -0.30893509188887674 -0.3070932135544732 … 0.3146450876734012 0.3164640331573624; -0.3051902857142649 -0.30337073400296327 … 0.31083106687075807 0.31262796371583534], [1.1921974630073207 1.1946538368553943 … -1.1719013209951015 -1.169420998897007; 1.1905381755462854 1.192991130639956 … -1.17027028148375 -1.167793411470972; … ; 0.40506320466390167 0.40589778676428917 … -0.3981673500920885 -0.3973246312901892; 0.4001531661808696 0.40097763176071666 … -0.3933409008142252 -0.39250839716319635])

In [10]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.


stl = load("crystal.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

2142

In [11]:
# Calculating and storing the vectors parallel to each face, e2 = V2 - V1 and e3 = V3 - V1, in preparation for hte Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

e2s = vertices[getindex.(indices, 2)] - vertices[getindex.(indices, 1)]
e3s = vertices[getindex.(indices, 3)] - vertices[getindex.(indices, 1)]

2142-element Vector{Point{3, Float32}}:
 [0.20006466, -0.12500381, 0.021842957]
 [-0.086564064, -0.24712181, 0.03823471]
 [-0.13139248, 0.22102165, -0.017211914]
 [0.11132717, -0.22167778, 0.014579773]
 [-0.1242342, -0.2677498, 0.004627228]
 [0.13139248, -0.22102165, 0.017211914]
 [-0.22294426, 0.20596504, 0.0055160522]
 [0.24161053, 0.0130290985, 0.010730743]
 [-0.16730404, 0.17181778, 0.01745224]
 [-0.32619762, -0.06916809, -0.0008392334]
 ⋮
 [0.28863525, -0.07749939, -0.0007972717]
 [-0.2492981, 0.23089218, -0.0039901733]
 [0.11425209, -0.24721527, 0.021465302]
 [-0.31781864, 0.21372223, -0.01871872]
 [0.26812553, -0.19742012, 0.006767273]
 [-0.03742695, 0.2215786, -0.019702911]
 [-0.28863525, 0.07749939, 0.0007972717]
 [-0.26812553, 0.19742012, -0.006767273]
 [0.050290108, 0.3967991, -0.016407013]

In [12]:
# Determining the coordinates of the points within our sample used for the Monte Carlo approximation of the volume integral.

const n_mc = 1
mc_coords = zeros(n_mc, 3)
# For this test, only one sample point used (at midpoint of coordinates).
max_coord = [maximum(getindex.(vertices, 1)), maximum(getindex.(vertices, 2)), maximum(getindex.(vertices, 3))]
min_coord = [minimum(getindex.(vertices, 1)), minimum(getindex.(vertices, 2)), minimum(getindex.(vertices, 3))]
mc_coords[1, :] = (min_coord + max_coord) / 2

3-element Vector{Float32}:
 16.6475
 20.065285
 47.947563

In [14]:
# Setting the (estimated) parameters of the sample.

# The number density of the sample in cm^-3.
const n = 1e23
# The reference absorption cross section at 25.3 meV in cm^2.
const axs_ref = 1e-23
const en_ref = 25.3

25.3

In [ ]:
# Defining the function that calculates the absorption cross sections for the inputted energy.

"""
Determines the absorption cross section (axs) for the inputted energy based on the absorption of the sample at a known, reference energy.

Parameters
----------
en (float): Energy in meV.

Returns
-------
axs (float): Absorption cross section in cm^2.
"""
function axs_calc(en :: Float64)
    return axs_ref * sqrt(en_ref / en)
end

@benchmark axs_calc(en_i, axs_ref, en_ref)

BenchmarkTools.Trial: 10000 samples with 1000 evaluations per sample.
 Range (min … max):  4.200 ns … 45.000 ns  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.600 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   6.198 ns ±  3.060 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▆█▆▅▄▁ ▂              ▂▁▁▆▁▂▁▂▁   ▂                        ▂
  ██████▅█▇▆▅▃▁▄▁▁▃▄▁▁▄▅█████████▅▄▄█▇▇▇▇▆▇▆▆▆▆▃▅▄▁▁▃▃▁▁▄▁▁▆ █
  4.2 ns       Histogram: log(frequency) by time     17.5 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [29]:
# Calculating the pre-scattering absorption cross section.

const axsi = axs_calc(en_i, axs_ref, en_ref)
p_i

2142-element Vector{SVector{3, Float64}}:
 [-0.0, 0.029187999067848712, 0.1670383411551405]
 [-0.0, 0.05109174199389585, 0.33022046133668204]
 [0.0, -0.022999694689859133, -0.29534370799713944]
 [-0.0, 0.01948245414553227, 0.296220469408189]
 [-0.0, 0.006183206927925338, 0.35778492255906974]
 [-0.0, 0.022999694689859133, 0.29534370799713944]
 [0.0, 0.007370912792893684, -0.27522407259357695]
 [0.0, 0.014339127030712264, -0.017410340694417967]
 [0.0, 0.023320834043906367, -0.22959424834351622]
 [0.0, -0.0011214390141332023, 0.09242696456483283]
 ⋮
 [0.0, -0.0010653670634265422, 0.103559795505137]
 [0.0, -0.005331932767196953, -0.3085333600383652]
 [-0.0, 0.02868335151148877, 0.33034534886325595]
 [0.0, -0.025013187465234652, -0.2855897372992127]
 [-0.0, 0.009042876413965005, 0.26380578444967523]
 [0.0, -0.026328329581809046, -0.29608793570651876]
 [0.0, 0.0010653670634265422, -0.103559795505137]
 [0.0, -0.009042876413965005, -0.26380578444967523]
 [0.0, -0.021924132726304106, -0.5302291

In [ ]:
# Creating the function to determine the length of the paths the neutrons take within the sample.
# Not broadcasted.

"""
Calculates the distance between a given point in the sample (origin) and a triangular face of the mesh that describes the surface. 
Iterates through each face to determine which one is intersected.
Exploits the method described in 'Fast, Minimum Storage Ray-Triangle Intersection' by Moller and Trumbore.

Parameters
----------
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
d (3-vector with float elements): Direction vector.
ps (n_faces-vector of 3-vectors with float elements): Array containing p = d x e3 for each face.
dets (n_faces-vector with float elements): Array containing det = p.e2 = (d x e3).e2 for each face.
origin (3-vector with float elements): Coordinates of scattering sites.
vertices (n_faces-vector of 3-vectors with float elements): Vertices of triangular faces.
indices (n_faces-vector of 3-vectors with integer elements): Indices describing which vertices form which triangles.

Returns
-------
path_length (float): Distance between origin and the surface the neutron path intersects, in units of the .stl file.
"""
function len_calc(
    e2s :: Vector{Point{3}}, 
    e3s :: Vector{Point{3}}, 
    d :: SVector{3, Float64}, 
    ps :: AbstractVector{<:AbstractVector{Float64}}, 
    dets :: Vector{Float64}, 
    origin :: Vector{Float64}, 
    vertices :: Vector{Point{3}}, 
    indices :: AbstractVector
)
    # If det = p.e2 = (d x e3).e2 = 0, the path is parallel to the triangular face, so it can never intersect it.
    # Keeping only positive determinants, equivalent to triangular faces in the forward direction.
    idx = findall(dets .> 1e-10)
    # Iterating through each valid face.
    @inbounds for j in idx
        # Calculating t = origin - V1 and q = t x e2 required for the MT algorithm.
        t = origin - vertices[indices[j][1]]
        q = cross(t, e2s[j])
        # Calculating the barycentric coordinates, (u,v), of the intersection.
        u = (1 / dets[j]) * (dot(ps[j], t))
        v = (1 / dets[j]) * (dot(q, d))
        # Determining whether the intersection point lies within the triangle.
        if v ≥ 0 && u ≥ 0 && (u + v) ≤ 1
            # Neutron's path described by r(λ) = origin + λd.
            λ = (1 / dets[j]) * dot(q, e3s[j])
            # Determining the path length based on λ and the magnitude of the inputted direction vector, d.
            path_length = abs(λ) * norm(d)
            return path_length
        end
    end
end

@benchmark len_calc(e2s, e3s, -ki, p_i, det_i, mc_coords[1,:], vertices, indices)

BenchmarkTools.Trial: 10000 samples with 6 evaluations per sample.
 Range (min … max):   6.633 μs … 159.933 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     10.317 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   13.063 μs ±   7.227 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▇▅▂ ▁▆▇█▇▆▄▃▂▂▁ ▁▂▂▂▁              ▂▃▄▄▄▃▃▁▁                 ▂
  ███▇██████████████████▇▆▆▆▄▄▅▅▄▅▅▆████████████▇▇▆▇▇▇▇▆▇▇▇▆▇▇ █
  6.63 μs       Histogram: log(frequency) by time        35 μs <

 Memory estimate: 9.57 KiB, allocs estimate: 11.

In [33]:
# Defining a function to pre-calculate p = d x e3 and determinant = p.e2 = (d x e3).e2 required for the MT Algorithm.

"""
Calculates p = d x e3 and determinant = p.e2 = (d x e3).e2 required for the MT Algorithm.

Parameters
----------
d (3-vector with float elements): Direction vector.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.

Returns
-------
ps (n_faces-vector of 3-vectors with float elements): Array containing p = d x e3 for each face.
dets (n_faces-vector with float elements): Array containing det = p.e2 = (d x e3).e2 for each face.
"""
# function pdet_calc(d :: SVector{3, Float64}, e2s :: AbstractVector{<:Point{3}}, e3s :: AbstractVector{<:Point{3}})
function pdet_calc(d :: SVector{3, Float64}, e2s :: Vector{Point{3}}, e3s :: Vector{Point{3}})
    # Pre-allocating these vectors.
    dets = Vector{Float64}(undef, n_faces)
    ps = Vector{SVector{3, Float64}}(undef, n_faces)
    # @inbounds is used to remove checks on the index i as we are sure of the sizes of our arrays.
    # @simd is used to vectorize and speed up the loop.
    @inbounds @simd for i in 1:n_faces
        # Calculating cross products, p = d x e3, for the direction vector, d, and for each face.
        ps[i] = cross(d, e3s[i])
        # Calculating the determinant = p.e2 = (d x e3).e2 for the direction vector, d, and for each face.
        dets[i] = dot(ps[i], e2s[i])
    end
    return ps, dets
end

@benchmark pdet_calc(-ki, e2s, e3s)

BenchmarkTools.Trial: 10000 samples with 6 evaluations per sample.
 Range (min … max):   5.517 μs …  39.699 ms  ┊ GC (min … max):  0.00% … 99.86%
 Time  (median):      9.842 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   36.250 μs ± 496.101 μs  ┊ GC (mean ± σ):  26.27% ±  5.78%

  █▆                                                            
  ██▇▄▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▂▂▂▂▁▁▁▁▂▂▃▅▆▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂ ▃
  5.52 μs         Histogram: frequency by time         72.2 μs <

 Memory estimate: 67.23 KiB, allocs estimate: 8.

In [39]:
# The pre-scattering neutron path lengths are dependent only on the MC coordinates.
# They can, therefore, be calculated and stored.

p_i, det_i = pdet_calc(-ki, e2s, e3s)
len_i = zeros(n_mc)
for i in 1:n_mc
    len_i[i] = len_calc(e2s, e3s, -ki, p_i, det_i, mc_coords[i, :], vertices, indices)
end
len_i

1-element Vector{Float64}:
 3.8700511152257744

In [47]:
# Defining the function that will calculate the attenuation factor given a certain energy bin and wavevector.

"""
Calculates the attenuation factor given a set initial and final energy and wavevector.

Parameters
----------
ki (3-vector with float elements): Pre-scattering wavevector of neutron, in Angstrom^-1.
kf (3-vector with float elements): Post-scattering wavevector of neutron, in Angstrom^-1.
en_f (float): Post-scattering energy of neutron, in meV.
vertices (n_faces-vector of 3-vectors with float elements): Vertices of the triangular faces.
indices (n_faces-vector of 3-vectors with integer elements): Indices describing which vertices correspond to which triangles.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
mc_coords (n_mc-vector of 3-vectors with float elements): Coordinates of sample points used in MC method.
len_i (n_mc-vector of float elements): Pre-scattering path length of neutron, in units of .stl file.

Returns
-------
atten_calc (float): Attenuation factor.
"""
function atten_calc(
    ki :: SVector{3, Float64}, 
    kf :: SVector{3, Float64}, 
    en_f :: Float64, 
    vertices :: AbstractVector{Point{3}}, 
    indices :: AbstractVector, 
    e2s :: Vector{Point{3}}, 
    e3s :: Vector{Point{3}}, 
    mc_coords :: AbstractMatrix{Float64}, 
    len_i :: AbstractVector{Float64}
)
    # Calculating the absorption cross section after the neutron scatters.
    axsf = axs_calc(en_f, axs_ref, en_ref)
    # Setting up the Moller-Trumbore algorithm for ray-triangle intersections.
    p_f, det_f = pdet_calc(kf, e2s, e3s)
    atten = 0
    for i in 1:n_mc
        # Calculating the path length, len_f, at this sample point.
        len_f = len_calc(e2s, e3s, kf, p_f, det_f, mc_coords[i, :], vertices, indices)
        # Adding the attenuation factor contribution from this sample point to A.
        atten += (1 / n_mc) * exp(-n * axsi * len_i[i]) * exp(-n * axsf * len_f)
    end
    return atten
end

@benchmark atten_calc(ki, SVector{3}([kx[1,1], ky[1,1], kz[1,1]]), ef_bins[1], vertices, indices, e2s, e3s, mc_coords, len_i)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  10.900 μs … 206.321 ms  ┊ GC (min … max):  0.00% … 99.95%
 Time  (median):     64.100 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   87.483 μs ±   2.063 ms  ┊ GC (mean ± σ):  23.57% ±  1.00%

                ▆█▂                                             
  █▃▄▂▂▁▁▁▁▁▁▁▃▇███▆▅▄▄▄▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  10.9 μs         Histogram: frequency by time          216 μs <

 Memory estimate: 76.40 KiB, allocs estimate: 22.

In [ ]:
# Defining the function that calculates the grid of attenuation factors.
# Dense arrays.

"""
Calculates the attenuation factor for every non-zero, non-NaN, signal and stores in a grid of detector against energy bin.

Parameters
----------
data (n_bins x n_detectors matrix with float elements): Neutron signal measured at different detectors for different energy bins.
kx (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in x direction, in Angstrom^-1.
ky (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in y direction, in Angstrom^-1.
kz (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in z direction, in Angstrom^-1.
ki (3-vector with float elements): Pre-scattering neutron wavevector, in Angstrom^-1.
en_i (float): Pre-scattering neutron energy, in meV.
ef_bins (n_bins-vector with float elements): Post-scattering neutron energy of each bin, in meV.
vertices (n_faces-vector of 3-vectors with float elements): Vertices of the triangular faces.
indices (n_faces-vector of 3-vectors with integer elements): Indices describing which vertices correspond to which triangles.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
mc_coords (n_mc-vector of 3-vectors with float elements): Coordinates of sample points used in MC method.
len_i (n_mc-vector with float elements): Pre-scattering path length of neutron, in units of .stl file.

Returns
-------
A_grid (n_bins x n_detectors matrix of floats): Attenuation factor grid.
"""
function a_grid_calc(
    data :: AbstractMatrix{Float64}, 
    kx :: AbstractMatrix{Float64}, 
    ky :: AbstractMatrix{Float64}, 
    kz :: AbstractMatrix{Float64}, 
    ki :: SVector{3, Float64}, 
    ef_bins :: SVector{n_bins, Float64}, 
    vertices :: AbstractVector{<:Point{3}}, 
    indices :: AbstractVector, 
    e2s :: Vector{Point{3}}, 
    e3s :: Vector{Point{3}}, 
    mc_coords :: AbstractMatrix{Float64}, 
    len_i :: AbstractVector{Float64}
)
    A_grid = zeros(n_bins, n_detectors)
    # Determining the locations in which the signal is either NaN or 0 as we don't want to calculate atten there.
    idx = findall(.~((data .== 0) .| (isnan.(data))))
    # Skipping checks on array lengths using @inbounds.
    @inbounds for I in idx
        kf = [kx[I], ky[I], kz[I]]
        A_grid[I] = atten_calc(ki, SVector{3}(kf), ef_bins[I[1]], vertices, indices, e2s, e3s, mc_coords, len_i)
    end
    return A_grid
end

@benchmark a_grid_calc(data, kx, ky, kz, ki, ef_bins, vertices, indices, e2s, e3s, mc_coords, len_i)

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 8.263 s (19.50% GC) to evaluate,
 with a memory estimate of 23.33 GiB, over 4767745 allocations.

In [57]:
# Testing the time taken to output this grid of attenuation factors.

A_grid = a_grid_calc(data, kx, ky, kz, ki, ef_bins, vertices, indices, e2s, e3s, mc_coords, len_i)
display(A_grid)
display(A_grid[160,6])

320×98304 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 ⋮                        ⋮              ⋱                 ⋮              
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0 

1.0463744196042372e-6

In [58]:
# Replacing all the NaNs with 0 to allow for sparse matrix conversion.

data_copy = copy(data)
data_copy .= ifelse.(isnan.(data_copy), 0, data)
sdata = sparse(data_copy)

320×98304 SparseMatrixCSC{Float64, Int64} with 317849 stored entries:
⎡⣷⣤⣇⣾⣮⣴⣿⣿⣼⣦⣗⣷⣷⣷⣴⣧⣷⣴⣦⣼⣆⣛⣽⣖⣺⣶⣀⣔⣰⣂⣠⣶⣕⣶⣾⣧⣱⣔⣷⣖⎤
⎣⣿⢉⣛⢿⣿⣿⣿⣿⣿⣻⣿⣿⣿⣿⡿⣿⡻⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⣿⣿⢿⡿⣿⢿⣿⣿⣿⎦

In [59]:
# Defining the function that calculates the grid of attenuation factors.
# Sparse arrays.

"""
Calculates the attenuation factor for every non-zero, non-NaN, signal and stores in a grid of detector against energy bin.

Parameters
----------
data (n_bins x n_detectors matrix with float elements): Neutron signal measured at different detectors for different energy bins.
kx (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in x direction, in Angstrom^-1.
ky (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in y direction, in Angstrom^-1.
kz (n_bins x n_detectors matrix with float elements): Post-scattering neutron wavevector component in z direction, in Angstrom^-1.
ki (3-vector with float elements): Pre-scattering neutron wavevector, in Angstrom^-1.
en_i (float): Pre-scattering neutron energy, in meV.
ef_bins (n_bins-vector with float elements): Post-scattering neutron energy of each bin, in meV.
vertices (n_faces-vector of 3-vectors with float elements): Vertices of the triangular faces.
indices (n_faces-vector of 3-vectors with integer elements): Indices describing which vertices correspond to which triangles.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
mc_coords (n_mc-vector of 3-vectors with float elements): Coordinates of sample points used in MC method.
len_i (n_mc-vector with float elements): Pre-scattering path length of neutron, in units of .stl file.

Returns
-------
A_grid (n_bins x n_detectors matrix with float elements): Attenuation factor grid.
"""
function a_grid_calc(
    sdata :: SparseMatrixCSC{Float64, Int}, 
    kx :: AbstractMatrix{Float64}, 
    ky :: AbstractMatrix{Float64}, 
    kz :: AbstractMatrix{Float64}, 
    ki :: SVector{3, Float64}, 
    ef_bins :: SVector{n_bins, Float64}, 
    vertices :: AbstractVector{<:Point{3}}, 
    indices :: AbstractVector, 
    e2s :: AbstractVector{<:Point{3}}, 
    e3s :: AbstractVector{<:Point{3}}, 
    mc_coords :: AbstractMatrix{Float64}, 
    len_i :: AbstractVector{Float64}
    ) :: SparseMatrixCSC{Float64, Int}
    A_grid = spzeros(n_bins, n_detectors)
    # Iterating through the locations in which the signal is not 0 and not NaN.
    # ... splits the tuple so it can be zipped together.
    idx = zip(findnz(sdata)[1:2]...)
    # Skipping checks on array lengths using @inbounds.
    @inbounds for (i, j) in idx
        kf = [kx[i, j], ky[i, j], kz[i, j]]
        A_grid[i, j] = atten_calc(ki, SVector{3, Float64}(kf), ef_bins[i], vertices, indices, e2s, e3s, mc_coords, len_i)
    end
    return A_grid
end

@benchmark a_grid_calc(sdata, kx, ky, kz, ki, ef_bins, vertices, indices, e2s, e3s, mc_coords, len_i)

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 9.882 s (10.60% GC) to evaluate,
 with a memory estimate of 23.11 GiB, over 4767792 allocations.

In [60]:
# Testing the time taken to output this grid of attenuation factors.

A_grid = a_grid_calc(sdata, kx, ky, kz, ki, ef_bins, vertices, indices, e2s, e3s, mc_coords, len_i)
display(A_grid)
display(A_grid[160,6])

320×98304 SparseMatrixCSC{Float64, Int64} with 317849 stored entries:
⎡⣷⣤⣇⣾⣮⣴⣿⣿⣼⣦⣗⣷⣷⣷⣴⣧⣷⣴⣦⣼⣆⣛⣽⣖⣺⣶⣀⣔⣰⣂⣠⣶⣕⣶⣾⣧⣱⣔⣷⣖⎤
⎣⣿⢉⣛⢿⣿⣿⣿⣿⣿⣻⣿⣿⣿⣿⡿⣿⡻⣿⣿⣿⣿⣿⣿⣿⢿⣿⣿⣿⣿⣿⣿⣿⣿⢿⡿⣿⢿⣿⣿⣿⎦

1.0463744196042372e-6